In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cp -r drive/MyDrive/ALBEF/* .

In [3]:
%cd /content/data

/content/data


In [4]:
# !unzip -q test2015.zip & unzip -q train2014.zip & unzip -q val2014.zip
!unzip -q test2015.zip

In [5]:
%cd /content

/content


In [6]:
!pip install transformers==4.25.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 111.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.3
    Uninstalling transformers-4.51.3:
      Successfully uninstalled transformers-4.51.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.25.1 which is incompatible.


In [7]:
!pip install ruamel.yaml==0.17.*

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.7/113.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 19.3 MB/s eta 0:00:00


In [8]:
import argparse
import os
import ruamel.yaml as yaml
import numpy as np
import random
import time
import datetime
import json
from pathlib import Path
import subprocess
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.backends.cudnn as cudnn
import torch.distributed as dist
from transformers import AutoTokenizer

from models.albef_blip2 import ALBEF_BLIP2
from models.vit import interpolate_pos_embed
from models.tokenization_bert import BertTokenizer

import utils
from dataset.utils import save_result
from dataset import create_dataset, create_sampler, create_loader, vqa_collate_fn

from scheduler import create_scheduler
from optim import create_optimizer

/usr/local/lib/python3.11/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [9]:
%load_ext autoreload
%autoreload 2

In [10]:
args = argparse.Namespace()
args.config = './configs/VQA_ALBEF_BLIP2.yaml'
# args.checkpoint = './ALBEF_4M.pth'
args.checkpoint = './drive/MyDrive/ALBEF_checkpoint/albef_blip2/checkpoint_07.pth'
args.output_dir = './output/vqa_albef_blip2'
args.text_encoder = 'bert-base-uncased'
args.text_decoder = "google-t5/t5-base"
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
args.seed = 42

config = yaml.load(open(args.config, 'r'), Loader=yaml.Loader)

In [11]:
args.result_dir = os.path.join(args.output_dir, 'result')

Path(args.output_dir).mkdir(parents=True, exist_ok=True)
Path(args.result_dir).mkdir(parents=True, exist_ok=True)

yaml.dump(config, open(os.path.join(args.output_dir, 'config.yaml'), 'w'))

In [12]:
datasets = create_dataset('vqa', config)
train_loader, test_loader = create_loader(datasets,[None, None],
                                          batch_size=[config['batch_size_train'],config['batch_size_test']],
                                          num_workers=[4,4],is_trains=[True, False],
                                          collate_fns=[vqa_collate_fn,None])
tokenizer_bert = BertTokenizer.from_pretrained(args.text_encoder)
tokenizer_t5 = AutoTokenizer.from_pretrained(args.text_decoder)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [13]:
device = torch.device(args.device)
model = ALBEF_BLIP2(config = config, text_encoder=args.text_encoder, text_decoder=args.text_decoder, tokenizer = tokenizer_t5)
model.to(args.device)
checkpoint = torch.load(args.checkpoint, map_location = 'cpu')
state_dict = checkpoint['model']
pos_embed_reshaped = interpolate_pos_embed(state_dict['visual_encoder.pos_embed'], model.visual_encoder)
state_dict['visual_encoder.pos_embed'] = pos_embed_reshaped

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [14]:
new_state_dict = OrderedDict()
for k in state_dict:
  if 'bert' in k:
    new_k = k.replace('bert.', '')
    new_state_dict[new_k] = state_dict[k]
  else:
    new_state_dict[k] = state_dict[k]

In [15]:
msg = model.load_state_dict(new_state_dict, strict = False)
print('load checkpoint from %s' % args.checkpoint)
print('Missing keys: %s' % str(msg.missing_keys))
print('Unexpected keys: %s' % str(msg.unexpected_keys))

load checkpoint from ./drive/MyDrive/ALBEF_checkpoint/albef_blip2/checkpoint_07.pth
Missing keys: []
Unexpected keys: []


In [16]:
for name, param in model.named_parameters():
    if not name.startswith('text_decoder'):
      param.requires_grad = False

In [17]:
arg_opt = utils.AttrDict(config['optimizer'])
optimizer = create_optimizer(arg_opt, model)
arg_sche = utils.AttrDict(config['scheduler'])
lr_scheduler, _ = create_scheduler(arg_sche, optimizer)

In [18]:
start_epoch = 0
max_epoch = config['scheduler']['epochs']
warmup_steps = config['scheduler']['warmup_epochs']

In [19]:
def train(model, data_loader, optimizer, tokenizer_bert, tokenizer_t5, epoch, warmup_steps, device, scheduler, config):
    # train
    model.train()

    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter('lr', utils.SmoothedValue(window_size=1, fmt='{value:.6f}'))
    metric_logger.add_meter('loss', utils.SmoothedValue(window_size=1, fmt='{value:.4f}'))

    header = 'Train Epoch: [{}]'.format(epoch)
    print_freq = 50
    step_size = 100
    warmup_iterations = warmup_steps*step_size

    for i,(image, question, answer, weights, n) in enumerate(metric_logger.log_every(data_loader, print_freq, header)):
        image, weights = image.to(device,non_blocking=True), weights.to(device,non_blocking=True)
        question_bert = tokenizer_bert(question, padding='longest', truncation=True, max_length=25, return_tensors="pt").to(device)
        question_t5 = ["question: " + q for q in question]
        question_t5 = tokenizer_t5(question_t5, padding='longest', return_tensors="pt").to(device)
        answer_input = tokenizer_t5(answer, padding='longest', return_tensors="pt")['input_ids'].to(device)


        loss = model(image, question_bert, question_t5, k = n, answer = answer_input)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        metric_logger.update(loss=loss.item())
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])

        if epoch==0 and i%step_size==0 and i<=warmup_iterations:
            scheduler.step(i//step_size)

    # gather the stats from all processes
    print("Averaged stats:", metric_logger.global_avg())
    return {k: "{:.3f}".format(meter.global_avg) for k, meter in metric_logger.meters.items()}

In [ ]:
!mkdir -p drive/MyDrive/ALBEF/output/vqa_albef_blip2

In [ ]:
for epoch in range(start_epoch, max_epoch):
  if epoch > 0:
    lr_scheduler.step(epoch + warmup_steps)
  train_stats = train(model, train_loader, optimizer, tokenizer_bert, tokenizer_t5, epoch, warmup_steps, device, lr_scheduler, config)

  log_stats = {**{f'train_{k}': v for k, v in train_stats.items()}, 'epoch': epoch}

  with open(os.path.join(args.output_dir, "log.txt"), "a") as f:
    f.write(json.dumps(log_stats) + "\n")

  save_obj = {
      'model': model.state_dict(),
      'optimizer': optimizer.state_dict(),
      'config': config,
      'epoch': epoch,
  }
  torch.save(save_obj, os.path.join(args.output_dir, 'checkpoint_%02d.pth' % (epoch)))
  subprocess.run(['cp', '-r', os.path.join(args.output_dir, 'checkpoint_%02d.pth' % (epoch)),
                  os.path.join('drive/MyDrive/ALBEF', args.output_dir)])

/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


Train Epoch: [0]  [    0/20565]  eta: 7:54:23  lr: 0.000010  loss: 0.9828  time: 1.3841  data: 1.0080  max mem: 10205


/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


Train Epoch: [0]  [   50/20565]  eta: 1:50:24  lr: 0.000010  loss: 0.9109  time: 0.3058  data: 0.0002  max mem: 10205
Train Epoch: [0]  [  100/20565]  eta: 1:46:50  lr: 0.000010  loss: 0.9344  time: 0.3009  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  150/20565]  eta: 1:44:59  lr: 0.000013  loss: 0.8469  time: 0.2992  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  200/20565]  eta: 1:43:50  lr: 0.000013  loss: 1.1332  time: 0.2957  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  250/20565]  eta: 1:43:00  lr: 0.000015  loss: 0.9069  time: 0.2983  data: 0.0003  max mem: 10921
Train Epoch: [0]  [  300/20565]  eta: 1:42:27  lr: 0.000015  loss: 0.7769  time: 0.2970  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  350/20565]  eta: 1:41:57  lr: 0.000018  loss: 0.7788  time: 0.2997  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  400/20565]  eta: 1:41:31  lr: 0.000018  loss: 0.6617  time: 0.2931  data: 0.0002  max mem: 10921
Train Epoch: [0]  [  450/20565]  eta: 1:41:04  lr: 0.000

/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


...

Train Epoch: [5]  [   50/20565]  eta: 1:51:28  lr: 0.000007  loss: 0.7864  time: 0.3019  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  100/20565]  eta: 1:48:25  lr: 0.000007  loss: 0.7490  time: 0.3043  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  150/20565]  eta: 1:46:25  lr: 0.000007  loss: 0.7973  time: 0.3026  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  200/20565]  eta: 1:45:10  lr: 0.000007  loss: 0.7442  time: 0.3001  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  250/20565]  eta: 1:44:10  lr: 0.000007  loss: 0.7691  time: 0.2927  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  300/20565]  eta: 1:43:38  lr: 0.000007  loss: 0.9832  time: 0.3041  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  350/20565]  eta: 1:43:09  lr: 0.000007  loss: 0.9019  time: 0.3017  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  400/20565]  eta: 1:42:40  lr: 0.000007  loss: 0.8659  time: 0.3051  data: 0.0002  max mem: 12342
Train Epoch: [5]  [  450/20565]  eta: 1:42:10  lr: 0.000

/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


Train Epoch: [6]  [    0/20565]  eta: 7:37:56  lr: 0.000004  loss: 0.8364  time: 1.3361  data: 0.9711  max mem: 12342
Train Epoch: [6]  [   50/20565]  eta: 1:49:55  lr: 0.000004  loss: 0.9275  time: 0.3016  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  100/20565]  eta: 1:47:03  lr: 0.000004  loss: 0.8240  time: 0.3082  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  150/20565]  eta: 1:45:51  lr: 0.000004  loss: 0.7761  time: 0.3000  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  200/20565]  eta: 1:44:35  lr: 0.000004  loss: 0.7439  time: 0.2924  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  250/20565]  eta: 1:44:01  lr: 0.000004  loss: 0.7238  time: 0.3009  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  300/20565]  eta: 1:43:17  lr: 0.000004  loss: 0.7335  time: 0.2970  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  350/20565]  eta: 1:42:47  lr: 0.000004  loss: 0.8702  time: 0.3002  data: 0.0002  max mem: 12342
Train Epoch: [6]  [  400/20565]  eta: 1:42:17  lr: 0.000

/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


Train Epoch: [7]  [    0/20565]  eta: 8:21:53  lr: 0.000002  loss: 0.7583  time: 1.4643  data: 1.0708  max mem: 12342
Train Epoch: [7]  [   50/20565]  eta: 1:53:07  lr: 0.000002  loss: 0.6799  time: 0.3070  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  100/20565]  eta: 1:49:29  lr: 0.000002  loss: 0.9934  time: 0.3133  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  150/20565]  eta: 1:47:09  lr: 0.000002  loss: 0.8889  time: 0.3030  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  200/20565]  eta: 1:45:29  lr: 0.000002  loss: 0.8046  time: 0.3015  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  250/20565]  eta: 1:44:27  lr: 0.000002  loss: 0.6573  time: 0.2997  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  300/20565]  eta: 1:43:39  lr: 0.000002  loss: 0.7849  time: 0.2942  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  350/20565]  eta: 1:42:55  lr: 0.000002  loss: 0.7278  time: 0.2928  data: 0.0002  max mem: 12342
Train Epoch: [7]  [  400/20565]  eta: 1:42:28  lr: 0.000

In [17]:
@torch.no_grad()
def evaluation(model, data_loader, tokenizer_bert, tokenizer_t5, device, config) :
    # test
    model.eval()

    metric_logger = utils.MetricLogger(delimiter="  ")
    header = 'Generate VQA test result:'
    print_freq = 50

    answer_list = [answer for answer in data_loader.dataset.answer_list]
    answer_input = tokenizer_t5(answer_list, padding='longest', return_tensors='pt').to(device)

    result = []

    for n, (image, question, question_id) in enumerate(metric_logger.log_every(data_loader, print_freq, header)):
        image = image.to(device,non_blocking=True)
        question_bert = tokenizer_bert(question, padding='longest', return_tensors="pt").to(device)
        question_t5 = ["question: " + q for q in question]
        question_t5 = tokenizer_t5(question_t5, padding='longest', return_tensors="pt").to(device)

        topk_ids, topk_probs = model(image, question_bert, question_t5, train=False, k=config['k_test'], answer = answer_input)
        for q, ques_id, topk_id, topk_prob in zip(question, question_id, topk_ids, topk_probs):
            ques_id = int(ques_id.item())
            _, pred = topk_prob.max(dim=0)
            result.append({"question_id":ques_id, "answer":data_loader.dataset.answer_list[topk_id[pred]]})

    return result

In [18]:
vqa_result = evaluation(model, test_loader, tokenizer_bert, tokenizer_t5, device, config)
result_file = save_result(vqa_result, args.result_dir, 'vqa')

Generate VQA test result:  [    0/27988]  eta: 1 day, 0:53:51    time: 3.2025  data: 0.6278  max mem: 10823
Generate VQA test result:  [   50/27988]  eta: 4:13:04    time: 0.4904  data: 0.0002  max mem: 10823
Generate VQA test result:  [  100/27988]  eta: 4:00:32    time: 0.4936  data: 0.0002  max mem: 11601
Generate VQA test result:  [  150/27988]  eta: 3:55:38    time: 0.4882  data: 0.0002  max mem: 11601
Generate VQA test result:  [  200/27988]  eta: 3:53:22    time: 0.4924  data: 0.0002  max mem: 11601
Generate VQA test result:  [  250/27988]  eta: 3:51:41    time: 0.4888  data: 0.0002  max mem: 11601
Generate VQA test result:  [  300/27988]  eta: 3:50:32    time: 0.4910  data: 0.0002  max mem: 11601
Generate VQA test result:  [  350/27988]  eta: 3:49:32    time: 0.4916  data: 0.0002  max mem: 11601
Generate VQA test result:  [  400/27988]  eta: 3:48:40    time: 0.4894  data: 0.0002  max mem: 11601
Generate VQA test result:  [  450/27988]  eta: 3:47:49    time: 0.4885  data: 0.0003

In [19]:
!cp -r output/vqa_albef_blip2 drive/MyDrive/ALBEF/output

In [21]:
vqa_result

[{'question_id': 262144000, 'answer': 'yes'},
 {'question_id': 262144001, 'answer': 'baseball'},
 {'question_id': 262144002, 'answer': 'yes'},
 {'question_id': 262144003, 'answer': 'no'},
 {'question_id': 262144004, 'answer': 'yes'},
 {'question_id': 262144005, 'answer': 'santa'},
 {'question_id': 1000, 'answer': 'wood'},
 {'question_id': 1001, 'answer': 'white'},
 {'question_id': 1002, 'answer': 'parking'},
 {'question_id': 524292000, 'answer': 'to see'},
 {'question_id': 524292001, 'answer': 'no'},
 {'question_id': 524292002, 'answer': 'no'},
 {'question_id': 524292003, 'answer': 'yes'},
 {'question_id': 524292004, 'answer': 'yes'},
 {'question_id': 131079000, 'answer': 'no'},
 {'question_id': 131079001, 'answer': 'yes'},
 {'question_id': 131079002, 'answer': 'taking off'},
 {'question_id': 131083000, 'answer': 'bench'},
 {'question_id': 131083001, 'answer': 'yes'},
 {'question_id': 131083002, 'answer': 'hat'},
 {'question_id': 131083003, 'answer': 'playing'},
 {'question_id': 131083